# jfinance クイックスタート

[jfinance](https://github.com/sgawa/jfinance) は、金融庁の電子開示システム **EDINET** に
提出された企業開示データを読み取る Python ライブラリです。

利用登録も API キーも要りません。API は yfinance に揃えてあるので、yfinance 向けに書いた
コードはほぼそのまま動きます。収録は **2016 年度以降**の有価証券報告書・半期報告書・
四半期報告書・大量保有報告書等です。

ドキュメント: <https://jfnc.org/ja/>


## インストール

In [ ]:
!pip install -q jfinance


In [ ]:
import jfinance as jf
import pandas as pd

# 財務諸表は 100 行ほどある。途中を省かせない
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.0f}")

jf.__version__


表示の設定がここでは効いてきます。財務諸表は 100 行ほどあり、pandas は既定で長い表の
途中を省くためです。次の設定ですべて表示します。


## 会社を指定する

指定には証券コード（`7203.T`）・EDINET コード（`E02144`）・ISIN（`JP3633400001`）・
ファンドコード（`G02925`）が使えます。4 つとも同じ会社に届きます。


In [ ]:
t = jf.Ticker("7203.T")

t.info["shortName"], t.info["edinetCode"], t.info["isin"]


### 会社の概要

In [ ]:
pd.Series({k: t.info[k] for k in (
    "shortName", "sector", "industry", "marketSegment", "listingStatus",
    "fiscalYearEnd", "fullTimeEmployees", "sharesOutstanding",
    "capitalMillionJpy", "returnOnEquity", "equityRatio", "trailingEps",
)}, name=t.info["symbol"]).to_frame()


## 財務諸表

`financials`・`balance_sheet`・`cashflow` は、行が科目、列が期末日（新しい順）の表を返します。
金額の単位は円です。

yfinance との違いが 2 つあります。

- **すべての年度を返します**（直近 4 期だけではありません）。
- EDINET に出所が無い科目も行としては残り、`NaN` が入ります。行の順序は yfinance の
  ままなので、行名で選ぶコードはそのまま動きます。


### 損益計算書（全科目）

In [ ]:
t.financials


### 貸借対照表（全科目）

In [ ]:
t.balance_sheet


### キャッシュ・フロー計算書（全科目）

In [ ]:
t.cashflow


### 期間の指定

四半期報告書は 2024 年 4 月に廃止され半期報告書に移行したため、四半期の値は 2023 年度で
止まります。


In [ ]:
# 年次・半期・四半期
for freq in ("yearly", "semiannual", "quarterly"):
    df = t.get_income_stmt(freq=freq)
    print(f"{freq:12} {df.shape[0]:3} x {df.shape[1]:2}   "
          f"{df.columns.min().date()} .. {df.columns.max().date()}")


### Yahoo に同名の科目が無い項目

経常利益・1 株当たり純資産・従業員数、および銀行業・保険業に固有の科目です。


In [ ]:
t.get_jp_financials()


### 提出どおりの全科目

`get_statements()` は、書類の表示の順と階層のまま返します（XBRL の表示リンクによります）。
`financials` はこれを yfinance の科目に畳んだものですが、こちらは畳みません。


In [ ]:
s = t.get_statements()

# 年度の列は整数。値を持つ行がいちばん多い表を選ぶ
years = [c for c in s.columns if isinstance(c, int)]
values = s[~s["Is Abstract"] & s[years].notna().any(axis=1)]
print(values["Role"].value_counts().head(6).to_string())


In [ ]:
role = values["Role"].value_counts().index[0]
print(role)

s[s["Role"] == role][["Label", "Local Name", "Depth", "Is Abstract"] + years[-3:]].head(30)


### 連結区分と訂正の反映

既定では、連結があれば連結を採り、訂正報告書を反映した値を返します。


In [ ]:
# 単体・提出時の値に切り替える（プロセス全体に効く）
jf.set_financials_basis(consolidation="standalone", version="as_filed")
standalone = t.financials.shape

jf.set_financials_basis()          # 既定へ戻す
print(standalone)


## 株主

日本の開示では株主の情報が複数の書類に分かれています。jfinance はこれらを統合せず、
書類ごとに分けて返します。


### 大株主の状況（有価証券報告書）

In [ ]:
t.major_shareholders


### 所有者別状況

In [ ]:
t.major_holders


### 大量保有報告書（5% ルール）

会社ではなく保有者が提出するもので、機関投資家に限らず事業会社や個人も含みます。


In [ ]:
h = jf.Ticker("8035.T")            # 東京エレクトロン
h.large_holders


In [ ]:
h.large_holder_transactions.head(10)


## 役員と報酬

役員は有価証券報告書で年 1 回開示されます。個別の報酬は連結報酬等が 1 億円以上の場合に
限って開示されるため、`officer_compensation` に現れない役員がほとんどです。


### 役員の状況

In [ ]:
t.officers


### 個別報酬（全年度）

In [ ]:
t.officer_compensation


### 役員区分ごとの報酬と、監査報酬

In [ ]:
display(t.officer_remuneration.head(8))
t.audit_fees.head(6)


## セグメント・従業員・提出書類

### セグメント別の売上高

In [ ]:
seg = t.segments.query("Metric == 'seg_revenue'")
seg.pivot_table(index="Segment Label", columns="Fiscal Year",
                values="Value", aggfunc="first").tail(10)


### 従業員の状況

In [ ]:
t.employees


### 提出書類

In [ ]:
# 種類コード 120 = 有価証券報告書
t.get_filings(types=["120"], limit=10)[
    ["Filing Date", "Title", "Fiscal Year", "Is Correction", "Document ID"]]


### 温室効果ガス排出量

XBRL に含まれるのは 2024 年度の有価証券報告書からなので、空の会社がまだ多くあります。
開示された実数であり、第三者が付ける ESG スコアではありません。


In [ ]:
jf.Ticker("9432.T").emissions      # NTT


## 会社を探す

### 検索

In [ ]:
pd.DataFrame(jf.Search("toyota").quotes)[["symbol", "shortname", "quoteType"]]


### 業種分類

In [ ]:
display(jf.JpSector.all())
jf.JpSector("automobiles-transportation-equipment").top_companies.head(10)


### スクリーナー

開示された数値で絞り込みます。書き方は yfinance の `screen` と同じです。

時価総額・PER・株価など株価から導く項目は、EDINET に株価が無いため扱えません。


In [ ]:
res = jf.edinet_screen(
    jf.EdinetQuery("and", [
        jf.EdinetQuery("gt", ["roe", 0.15]),
        jf.EdinetQuery("gt", ["equityRatio", 0.5]),
    ]),
    sortField="revenue",
    size=20,
)
print(res["total"], "社が該当")
pd.DataFrame(res["quotes"])[
    ["symbol", "shortName", "sector", "roe", "equityRatio", "revenue"]]


#### 指定できる項目

In [ ]:
q = jf.EdinetQuery("gt", ["roe", 0.2])
for group, fields in q.valid_fields.items():
    print(f"{group:12} {len(fields):3}  {', '.join(list(fields)[:6])}")


## 投資信託

1 期あたり 25 項目です。ファンドコード（`G` で始まる）か、上場していれば証券コードで
指定します。


In [ ]:
jf.Ticker("1306.T").get_fund_financials()


## 日付を指定して提出書類を見る

全社を横断して取得します。


In [ ]:
jf.FilingCalendar("2026-06-25").get_filings(types=["120"], limit=10)[
    ["Filing Date", "Title", "Document ID"]]


## 日本語で取得する

社名と業種は既定で英語です。

事業の内容、役員・株主の氏名は、EDINET に英語の原文が無いため、どちらの設定でも
日本語で返ります。


In [ ]:
jf.config.locale.lang = "ja-JP"

t = jf.Ticker("7203.T")
print(t.info["shortName"], "/", t.info["sector"], "/", t.info["industry"])


## 扱わないもの

株価・配当の履歴・株式分割・オプション・アナリスト予想・ニュース・決算発表予定・
ESG スコアは EDINET に存在しません。yfinance の対応する属性は**定義していない**ので、
参照すると空ではなく `AttributeError` になります。


In [ ]:
for name in ("history", "dividends", "splits", "news",
             "recommendations", "earnings_dates", "sustainability"):
    print(f"{name:16} {'present' if hasattr(t, name) else 'not defined'}")


## お読みください

**EDINET の閲覧期間が満了した書類への訂正報告書は取得できないため、反映できない場合が
あります。**古い年度ほど、訂正前の値が残っている可能性が高くなります。


In [ ]:
print(jf.NOTICE_CORRECTIONS)


---

出典と利用条件:

```text
出典：EDINET閲覧（提出）サイト（https://disclosure2.edinet-fsa.go.jp/）、
      PDL1.0（https://www.digital.go.jp/resources/open_data/public_data_license_v1.0）
EDINET閲覧（提出）サイト（https://disclosure2.edinet-fsa.go.jp/）をもとに jfinance 作成
```

同じ文言を、すべての応答の `X-JF-Notice` ヘッダでも返しています。
